## **Random Forest from Scratch**

### **Topic Roadmap**

**1. Generate a reproducible classification dataset**

**2. Implement row and feature sampling**

**3. Train sampled decision trees**

**4. Aggregate predictions by majority vote**

**5. Key revision notes**

## **1. Dataset**

A random forest is bagging of decision trees with random feature selection. The implementation below focuses on the sampling and voting mechanics rather than reproducing every optimization in scikit-learn.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
X, y = make_classification(
    n_samples=1200, n_features=8, n_informative=4, n_redundant=0,
    class_sep=1.2, random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## **2. Sampling Functions**

Each tree receives a bootstrap sample of rows and a random subset of columns. The sampled feature indices must be retained so test rows use the same columns.

In [3]:
def bootstrap_indices(n_rows, rng):
    return rng.integers(0, n_rows, size=n_rows)

def feature_indices(n_features, max_features, rng):
    count = max(1, int(max_features * n_features))
    return np.sort(rng.choice(n_features, size=count, replace=False))

## **3. Train the Custom Forest**

A shallow tree is fitted for every sampled dataset. Randomness enters through both bootstrap rows and selected features.

In [4]:
def fit_random_forest(X, y, n_estimators=50, max_features=0.7, max_depth=None, random_state=42):
    rng = np.random.default_rng(random_state)
    forest = []
    for _ in range(n_estimators):
        rows = bootstrap_indices(X.shape[0], rng)
        cols = feature_indices(X.shape[1], max_features, rng)
        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=int(rng.integers(0, 1_000_000)))
        tree.fit(X[rows][:, cols], y[rows])
        forest.append((tree, cols))
    return forest

In [5]:
def predict_forest(forest, X):
    tree_predictions = np.column_stack([tree.predict(X[:, cols]) for tree, cols in forest])
    return (tree_predictions.mean(axis=1) >= 0.5).astype(int)

custom_forest = fit_random_forest(X_train, y_train, n_estimators=75, max_features=0.7, max_depth=8, random_state=RANDOM_STATE)
y_pred = predict_forest(custom_forest, X_test)
print(f"Custom forest accuracy: {accuracy_score(y_test, y_pred):.3f}")

Custom forest accuracy: 0.917


## **4. Inspect One Tree’s Feature Subset**

The stored column indices show that different trees can see different feature spaces.

In [6]:
for tree_number, (tree, cols) in enumerate(custom_forest[:5], start=1):
    print(f"Tree {tree_number}: features {cols.tolist()}")

Tree 1: features [0, 1, 2, 5, 7]
Tree 2: features [0, 1, 2, 5, 6]
Tree 3: features [2, 4, 5, 6, 7]
Tree 4: features [0, 3, 4, 5, 7]
Tree 5: features [0, 1, 3, 5, 6]


### **Key Revision Notes**

- A random forest combines bootstrap row sampling, random feature selection, decision trees, and aggregation.
- Every tree must remember its selected columns.
- Classification predictions are aggregated by majority vote.
- Production implementations add optimized split search, parallelism, missing-value handling, and more diagnostics.